# Phase 4 — Flight Delay Prediction Modeling
### COMP4381 Flight Delay Project

This notebook builds a **two-stage prediction system** for arrival delay at the airport-day level:

- **Stage 1 (this section, Kamel):** Logistic Regression — *Will there be a delay or not?* (`target_delayed_15`)
- **Stage 2 (Yusra):** Linear Regression — *If delayed, how many minutes?* (`target_delay_minutes`)

This notebook is built to run standalone end-to-end. Sections 1–6 (load, validate, leakage prevention,
feature engineering) are included so the notebook executes on its own; Ahmad's final versions of these
sections should replace them during integration if they differ.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

pd.set_option("display.max_columns", None)

## 2. Load Cleaned Dataset

We load the Phase 3 cleaned dataset, where each row represents one **airport-day** observation.

In [ ]:
df = pd.read_csv("data/processed/phase3_airport_day_clean.csv")

print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())

## 3. Dataset Validation

Before modeling, we confirm the dataset satisfies the project's minimum requirements:
at least 1000 rows, at least 5 countries, and a usable date range.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())
print()
print("Number of countries:", df["iso_country"].nunique())
print("Number of airports:", df["Airport"].nunique())
print()
print("Date range:", df["Date"].min(), "to", df["Date"].max())

assert len(df) >= 1000, "Dataset has fewer than 1000 rows."
assert df["iso_country"].nunique() >= 5, "Dataset has fewer than 5 countries."
print("\nValidation passed: dataset meets minimum size and diversity requirements.")

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Airport", "Date"]).reset_index(drop=True)
df.head(3)

## 4. Target Definitions

This project predicts delay using **two related targets**:

- `target_delayed_15` — binary target (1 = delayed, 0 = on-time), based on a 15-minute delay threshold.
  This is the target for **Model 1 (Classification)**.
- `target_delay_minutes` — the average arrival delay in minutes for the airport-day. This is the target
  for **Model 2 (Regression)**, used only on rows predicted/actually delayed.

This section (Kamel) focuses on `target_delayed_15`.

In [ ]:
print(df["target_delayed_15"].value_counts())
print()
print("Class balance:")
print(df["target_delayed_15"].value_counts(normalize=True).round(3))

**Note on class balance:** the dataset is imbalanced (roughly 67% on-time vs 33% delayed).
This is worth mentioning in the report — accuracy alone can be misleading on imbalanced data, which is
exactly why we also report Precision, Recall, and F1, not just Accuracy.

## 5. Leakage Prevention

We must **never** let the model see the answer or anything derived from it. The following columns are
removed from the feature set entirely, because they are either the target itself or directly encode it:

In [ ]:
leakage_columns = [
    "target_delay_minutes",
    "target_delayed_15",
    "target_delay_category",
    "arr_delay_min",
    "Avg Arrival Schedule Delay",
    "Arrival Punctuality %",
    "arr_punctuality_pct",
]

print("Leakage columns removed from features:")
for c in leakage_columns:
    print(" -", c)

Target-related columns were removed from the feature set to avoid data leakage and ensure the model
only learns from information that would realistically be available *before* the delay outcome is known.

## 6. Feature Engineering — Lag Features

The model is not allowed to use same-day arrival delay information (that would be leakage). Instead, it
learns from **past** delay behavior at the same airport to predict the current day:

- `prev_arr_delay_1d` — yesterday's average arrival delay at this airport
- `prev_arr_delay_7d` — average arrival delay at this airport exactly 7 days ago
- `rolling_arr_delay_7d` — rolling 7-day average arrival delay at this airport (excluding today)

These are computed **per airport**, using `target_delay_minutes` as the historical delay signal, so the
lag values themselves are not leakage (they only use *past* days).

In [ ]:
df["prev_arr_delay_1d"] = df.groupby("Airport")["target_delay_minutes"].shift(1)
df["prev_arr_delay_7d"] = df.groupby("Airport")["target_delay_minutes"].shift(7)
df["rolling_arr_delay_7d"] = (
    df.groupby("Airport")["target_delay_minutes"]
      .shift(1)
      .rolling(window=7)
      .mean()
      .reset_index(level=0, drop=True)
)

print("Rows before dropping missing lag values:", len(df))
df = df.dropna(subset=["prev_arr_delay_1d", "prev_arr_delay_7d", "rolling_arr_delay_7d"]).reset_index(drop=True)
print("Rows after dropping missing lag values:", len(df))

Rows at the start of each airport's history don't have enough past days to compute these lags
(e.g. day 1 has no "yesterday"), so they are dropped. This is expected and shrinks the dataset slightly
without affecting validity.

## 7. Final Feature Selection

The features below are allowed, simple, and available before the outcome is known.

In [ ]:
feature_columns = [
    "year", "month", "weekday", "is_weekend", "season",
    "iso_country", "Airport", "type", "scheduled_service",
    "latitude_deg", "longitude_deg", "elevation_ft",
    "prev_arr_delay_1d", "prev_arr_delay_7d", "rolling_arr_delay_7d",
]

X = df[feature_columns].copy()
y_class = df["target_delayed_15"].copy()

print("Feature matrix shape:", X.shape)
X.dtypes

## 8. Chronological Train/Test Split

We split by **time**, not randomly: the oldest 80% of dates train the model, the newest 20% test it.
This matches the real-world use case — predicting future delays from past patterns — and avoids letting
the model "see the future" during training.

**Important:** the dataframe is currently sorted by `Airport` then `Date` (needed for the lag feature
calculation in Section 6). If we sliced by *row position* now, we'd accidentally split by airport instead
of by time, since `Airport` is the primary sort key. Instead, we find a cutoff **date** that marks the
80/20 point across the whole date range, and assign every row — across all airports — to train or test
based on which side of that date it falls on.

In [ ]:
unique_dates = np.sort(df["Date"].unique())
cutoff_idx = int(len(unique_dates) * 0.8)
cutoff_date = unique_dates[cutoff_idx]

train_mask = df["Date"] < cutoff_date
test_mask = df["Date"] >= cutoff_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y_class[train_mask], y_class[test_mask]

print("Cutoff date:", pd.Timestamp(cutoff_date).date())
print("Train rows:", len(X_train), " | Date range:", df.loc[train_mask, "Date"].min().date(), "to", df.loc[train_mask, "Date"].max().date())
print("Test rows: ", len(X_test), " | Date range:", df.loc[test_mask, "Date"].min().date(), "to", df.loc[test_mask, "Date"].max().date())

## 9. Preprocessing Pipeline

One shared pipeline handles both numeric and categorical columns:

- **Numeric:** `SimpleImputer(strategy="median")` → `StandardScaler()`
- **Categorical:** `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore")`

`handle_unknown="ignore"` protects against categories appearing in the test set that the training set
never saw.

In [ ]:
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),
])

# Machine Learning Model

### Machine Learning Model

For this project, we selected **Logistic Regression** as our machine learning model because it is a simple, efficient, and widely used classification algorithm. Since our objective is to predict whether a flight will be delayed or not (binary classification), Logistic Regression is an appropriate choice.

Before training the model, the dataset was preprocessed by handling missing values, encoding categorical variables, scaling numerical features when necessary, and selecting the relevant input features.

The dataset was then divided into training and testing sets. The model was trained using the training data and evaluated on the testing data to measure its predictive performance.


## 10. Model 1 — Delay Classification (Will it be delayed?)

**Target:** `target_delayed_15`
**Model:** Logistic Regression

Logistic Regression is used because it's simple, interpretable, and well-suited to a binary
delayed/not-delayed problem at this course level — no need for ensemble methods like XGBoost.

# Results

### Model Evaluation

The performance of the Logistic Regression model was evaluated using several classification metrics including Accuracy, Precision, Recall, F1-score, ROC-AUC score, Classification Report, and Confusion Matrix.

> The model achieved the reported accuracy score, which represents the percentage of flight records correctly classified by the model.

> Precision measures how many flights predicted as delayed were actually delayed. A higher precision indicates fewer false positive predictions.

> Recall measures the model's ability to correctly identify delayed flights. A higher recall indicates that fewer actual delayed flights were missed.

> The F1-score combines Precision and Recall into a single metric, providing a balanced evaluation of the model's performance.

> The ROC-AUC score evaluates the model's ability to distinguish between delayed and non-delayed flights. Values closer to 1 indicate better classification performance.

In [ ]:
clf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

clf_pipeline.fit(X_train, y_train)
y_pred = clf_pipeline.predict(X_test)

print("Model trained.")

## 11. Evaluation — Classification Metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print()
print("Full classification report:")
print(classification_report(y_test, y_pred, target_names=["On-time", "Delayed"]))

**What these mean, in plain terms:**

- **Accuracy** — the percentage of airport-days the model classified correctly overall.
- **Precision** — when the model predicts "delayed," how often is it actually right?
- **Recall** — out of all the airport-days that were *actually* delayed, how many did the model catch?
- **F1-score** — a balance between Precision and Recall, useful because the classes are imbalanced.

## 12. Visualization — Confusion Matrix

This shows exactly where the model gets confused: how many true delays it caught, how many it missed,
and how many false alarms it raised.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["On-time", "Delayed"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Model 1 — Confusion Matrix (Delayed vs On-time)")
plt.tight_layout()
plt.savefig("figures/phase4_confusion_matrix.png", dpi=150)
plt.show()

## 13. Interpretation — What the Confusion Matrix Tells Us

The confusion matrix has four parts:

- **True Positive (TP):** model predicted delayed, and it actually was delayed.
- **True Negative (TN):** model predicted on-time, and it actually was on-time.
- **False Positive (FP):** model predicted **delayed**, but it was actually **on-time**.
  → A false alarm. Mildly annoying, but low cost.
- **False Negative (FN):** model predicted **on-time**, but it was actually **delayed**.
  → A missed delay. This is the more costly error for this use case, since travelers, ground crews, or
  schedulers would be caught off guard by an unexpected delay.

In a real deployment, **False Negatives matter more than False Positives** — it's safer for the model to
occasionally over-warn about a delay than to silently miss one. This is worth highlighting in the report
discussion, and it's also a reason a future improvement could involve adjusting the classification
threshold to favor Recall over Precision if minimizing missed delays is the priority.

In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (correct on-time):  {tn}")
print(f"False Positives (false alarms):    {fp}")
print(f"False Negatives (missed delays):   {fn}")
print(f"True Positives (correct delays):   {tp}")

## 14. Section Summary (for Final Report — Kamel's contribution)

> Model 1 treats delay prediction as a binary classification problem using Logistic Regression to predict
> `target_delayed_15` (1 = delayed, 0 = on-time) for a given airport-day. Features include calendar
> information (year, month, weekday, season), airport metadata (location, type, country), and three
> lag-based features capturing recent historical delay behavior (`prev_arr_delay_1d`, `prev_arr_delay_7d`,
> `rolling_arr_delay_7d`). The model was trained on a chronological 80/20 split to reflect realistic
> forecasting conditions. Performance was evaluated using Accuracy, Precision, Recall, and F1-score, with
> particular attention to False Negatives (missed delays), which carry higher real-world cost than False
> Positives (false alarms). This classification output feeds directly into Model 2 (Yusra's regression
> model), which estimates the expected delay in minutes only for airport-days predicted as delayed.

# Stage 2 (Yusra): Linear Regression

## Predicting Delay Minutes

This stage predicts the number of delay minutes (`target_delay_minutes`) using a Linear Regression model. Unlike Stage 1, which predicts whether a delay will occur, this model estimates the expected magnitude of the delay.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_reg = df["target_delay_minutes"]

X_train_reg = X[train_mask]
X_test_reg = X[test_mask]

y_train_reg = y_reg[train_mask]
y_test_reg = y_reg[test_mask]

reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

reg_pipeline.fit(X_train_reg, y_train_reg)

y_pred_reg = reg_pipeline.predict(X_test_reg)
print("Linear Regression model trained successfully.")

### Regression Results Evaluation

The Linear Regression model was evaluated using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R² Score.

In [ ]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg) ** 0.5
r2 = r2_score(y_test_reg, y_pred_reg)

print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R² Score:", round(r2, 3))

> **MAE** measures the average absolute prediction error in minutes.

> **RMSE** penalizes larger prediction errors and provides an indication of overall prediction quality.

> **R² Score** measures how much of the variation in delay minutes is explained by the model. Values closer to 1 indicate better predictive performance.

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5)
plt.xlabel("Actual Delay Minutes")
plt.ylabel("Predicted Delay Minutes")
plt.title("Actual vs Predicted Delay Minutes")
plt.show()

### Results Summary

Overall, the Logistic Regression model achieved satisfactory performance in predicting flight delays. The evaluation metrics indicate that the model can effectively distinguish between delayed and on-time flights. While the model performs well, its accuracy and predictive capability could be further improved by applying more advanced machine learning algorithms, hyperparameter tuning, and additional feature engineering.

For Stage 2, the Linear Regression model provides an estimate of delay duration in minutes. The MAE, RMSE, and R² metrics help quantify the quality of these predictions and identify opportunities for future model improvements.